# Project 1: Data Cleaning

<span style="background-color:red;color:white;">SHOULD ALSO BE IN README CONTENT</span>

Submitted on October 12, 2025

NLP1000 S17
Group 12: 
| Member Name      | ID Number      |
| ------------- | ------------- |
| Alcantara, Van Asher | 12340898 |
| Aragon, Enrique | 12227811 |
| Clavano, Angelica (Jack) | 12206245 |
| Lozada, Job | 12307246 |


## Table of Contents:
```
1. Data Selection
1.1 Web Scraping
2. Data Cleaning and Segmentation (includes breakdown of steps)
3. Parallel Corpus Creation
4. AI Declaration
5. References
```

---

## Running this Notebook:
- Make sure you have at least Python 3.12.0 or 3.14.0 installed.
- See first code block under 1.1 for `pip requirements`.

---

## Folder Structure:
```
/nlp1000 --> root
└── /data
    └── lang.txt files --> raw webscraped data
    └── /cleaned
        └── (sentences)-lang-cleaned.txt --> cleaned sentence files
        └── lang-cleaned.txt --> cleaned verse files
└── /other --> other files, will be deleted before submission
└── .gitignore
└── main.ipynb --> MAIN SUBMISSION
└── README.md --> contains the same information as this markdown block.
└── all-languages-cleaned-verses.xlsx --> verses combined
└── parallel_corpora.xlsx --> parallel corpora (from all-languages-cleaned-verses.xlsx)
└── requirements.txt --> project dependencies for this project.
```

## 1. Data Selection

For all 16 languages, we used the books of the Gospel, which are Matthew, Mark, Luke, and John, as our corpora. We initially considered using the books of Genesis and Exodus as the corpora, but the translations for more niche languages typically only spanned the New Testament. All corpora were sourced from https://www.bible.com/. Later, a word count summary will be provided.

| Target Language      | Link to Source      | Comments
| ------------- | ------------- |  ------------- | 
| Spanish  | https://www.bible.com/versions/1076-jbs-biblia-del-jubileo | |
| Tagalog | https://www.bible.com/versions/177-tlab-ang-biblia | |
| English | https://www.bible.com/versions/3523-nrsvue-new-revised-standard-version-updated-edition-2021 | |
| Hiligaynon/Ilonggo | https://www.bible.com/versions/2190-mbbhil12-maayong-balita-nga-biblia-2012 | Listed as Ilonggo on the website |
| Bikol/Bikolano | https://www.bible.com/versions/890-mbbbik92-marahay-na-bareta-biblia |  |
| Waray | https://www.bible.com/versions/2198-mbbsam-samarenyo-meaning-based-bible-1984 | |
| Ilocano | https://www.bible.com/versions/782-ripv-ti-baro-a-naimbag-a-damag-biblia | |
| Cebuano | https://www.bible.com/versions/562-rcpv-ang-bag-ong-maayong-balita-biblia | |
| Kapampangan | https://www.bible.com/versions/1141-pmpv-ing-mayap-a-balita-biblia | |
| Pangasinense | https://www.bible.com/versions/2194-mbbpan83-maung-a-balita-biblia | |
| Yakan | https://www.bible.com/versions/1388-yakv-yakan | |
| Ivatan | https://www.bible.com/versions/1315-vtsp-ivatan | |
| Tausug  | https://www.bible.com/versions/1319-tsg-kitab-injil | |
| Yami  | https://www.bible.com/versions/2364-snt-seysyo-no-tao | |
| Tuwali Ifugao | https://www.bible.com/versions/2123-ifkwb-nan-kalin-apu-dios | Listed as "tuwali_ifugao" in project files |
| Masbateño/Masbatenyo | https://www.bible.com/versions/1222-msb-masbatenyo | Listed as "masbateno" in project files |
| TOTAL | 16 | |

### 1.1 Web Scraping

Using `webscraper.py`, we were able to extract all the Gospels in the languages listed in Section 1. First, the bible abbreviation and number is compiled, then for each of the chapters and their chapter ranges (mat, mrk, luk, jhn) -- which are all listed the same on Bible.com -- it will scrape for the text and compile them in the `/data` folder. This may take around 3-5 min. If on public wifi, expect around 25-40 minutes.

The raw data has been saved ahead with the `/data` folder under the following naming convention: `lang.txt`

In [1]:
# run the following commands one by one or run pip install -r requirements.txt
from bs4 import BeautifulSoup       # pip install beautifulsoup4
from pathlib import Path            # pip install pathlib
import pandas as pd                 # pip install pandas
                                    # pip install xlsxwriter
                                    # pip install ipykernel
                                    # pip install openpyxl 
                                    
from urllib.request import urlopen 
import re

Using information from the URLs of the Gospels, we compile all of them here. Note that all bibles hosted on Bible.com will have the same book codes.

In [2]:
languages = ["spanish", "tagalog", "english", "hiligaynon", "bikol", "waray", "ilocano", "cebuano", "kapampangan", "pangasinense", "yakan", "ivatan", "tausug", "yami", "tuwali_ifugao", "masbateno"]
bibleNumbers = ["1076", "177", "3523", "2190", "890", "2198", "782", "562", "1141", "2194", "1388", "1315", "1319", "2364", "2123", "1222"]
bibleAbbreviation = ["JBS", "TLAB", "NRSVUE", "MBBHIL12", "MBBBIK92", "MBBSAM", "RIPV", "RCPV", "PMPV", "MBBPAN83", "YAKV", "VTSP", "TSG", "SNT", "IFKWB", "MSB"]

bookCodes = ["mat", "mrk", "luk", "jhn"]
chapterRanges = {
    "mat": [1, 28],
    "mrk": [1, 16],
    "luk": [1, 24],
    "jhn": [1, 21]
}

Next, we began to webscrape using the following code. For all languages listed above, associated with their bible number and abbreviation, it will proceed to scrape every chapter of Matthew, Mark, Luke, and John.

All data from this webscraper is located in `/data` with the naming convention `lang.txt`

In [3]:
# this has been commented out so that it won't run every time. feel free to delete the data folder if you would like to rescrape.
# latest scrape: October 12, 2025
'''
# webscraper v4; adjusted to use 'Path' instead of 'os'
# instead of "!python webscraper_v3.py"
output_folder = "data"
data_folder = Path("data")
data_folder.mkdir(parents=True, exist_ok=True) 

# clear content of the file if it exists so it doesn't make copies
for lang in languages:
    file_path = data_folder / f"{lang}.txt"
    file_path.write_text("", encoding="utf-8")

for lang, bibleNumber, abbreviation in zip(languages, bibleNumbers, bibleAbbreviation):
  # if in the previous code block, languages, bibleNumbers, and/or bibleAbbreviation is "SKIP", just skip it
  if bibleNumber == "SKIP" or abbreviation == "SKIP":
        continue
  
  for bookCode, (start, end) in chapterRanges.items():
     for chapter in range(start, end + 1):
      urlPartOne = "https://www.bible.com/bible/"
      chapterNumber = str(chapter)
      bibleName = f".{abbreviation}"

      url = f"{urlPartOne}{bibleNumber}/{bookCode.upper()}.{chapterNumber}{bibleName}"
      print(f"Scraping URL: {url}")

      try: 
        page = urlopen(url)
        html = page.read().decode("utf-8")
        soup = BeautifulSoup(html, "html.parser")
        
        #IMPORTANT: THIS ONLY WORKS ON BIBLE.COM, for other sites just use inspect element then select the div/class containing the text so that it extracts just that
        text = soup.find('div', {'class': 'ChapterContent_reader__Dt27r'})

        if text:
          # excludes the pop-up notes
          for note in text.select('span.ChapterContent_note__YlDW0'):
            note.decompose()
          
          # excludes chapter headings
          for heading in text.select('span.ChapterContent_heading__xBDcs'):
            heading.decompose()
        
          # write to lang-specific file
          file_path = Path(output_folder) / f"{lang}.txt" # file_name = os.path.join(output_folder, f"{lang}.txt")

          with file_path.open("a", encoding="utf-8") as f: # with open(file_name, "a", encoding="utf-8") as f:
              f.write(text.get_text() + "\n") 
        else:
            print(f"No content found for {lang}, {bookCode}, chapter {chapter}")
      except Exception as e:
                  print(f"Error scraping {url}: {e}")'''

'\n# webscraper v4; adjusted to use \'Path\' instead of \'os\'\n# instead of "!python webscraper_v3.py"\noutput_folder = "data"\ndata_folder = Path("data")\ndata_folder.mkdir(parents=True, exist_ok=True) \n\n# clear content of the file if it exists so it doesn\'t make copies\nfor lang in languages:\n    file_path = data_folder / f"{lang}.txt"\n    file_path.write_text("", encoding="utf-8")\n\nfor lang, bibleNumber, abbreviation in zip(languages, bibleNumbers, bibleAbbreviation):\n  # if in the previous code block, languages, bibleNumbers, and/or bibleAbbreviation is "SKIP", just skip it\n  if bibleNumber == "SKIP" or abbreviation == "SKIP":\n        continue\n\n  for bookCode, (start, end) in chapterRanges.items():\n     for chapter in range(start, end + 1):\n      urlPartOne = "https://www.bible.com/bible/"\n      chapterNumber = str(chapter)\n      bibleName = f".{abbreviation}"\n\n      url = f"{urlPartOne}{bibleNumber}/{bookCode.upper()}.{chapterNumber}{bibleName}"\n      print(f"S

The text data as is will show up in the following format: 

```txt
Mateo 1

1An mga Ginikanan ni Jesu-Cristo(Lk. 3:23-38) 1Iyo ini an mga ginikanan ni Jesu-Cristo, na gikan ki...

Mateo 2

2An Pagdalaw kan mga Mago 1Namundag si Jesus sa Betlehem nin Judea, kan si Herodes an hade...
```

So...

```txt
(book)(space)(book number)
(new line)
(chapter)(title)(subtitle if any)(space)(line number)(content)
```

where the `(line number)(content)` repeats for the entire page, and then a new line is made for the next chapter to begin. This can be cleaned using Regular Expressions/regex.


## 2. Data Cleaning & Segmentation
Data Cleaning: *Remove document tags, unnecessary spaces, duplicated or repetitive terms, and unrelated symbols or keywords from the data.*

Segmentation: *Properly segment each paragraph into (a) verses and (b) sentences. An example is shown below.*

The following function, `divide_into_verses` takes the raw lang.txt files and divides them into verses. The following steps are as follows:
<span style="background-color:red;color:white;">STEPS HERE</span>

In [4]:
# changed all \t to |
def divide_into_verses(text):
    # WEIRD ENCODING: replace weird ‘ with ' accounts for letters before and after
    text = re.sub(r"(\w)‘(\w)", r"\1'\2", text, flags=re.MULTILINE)
    text = re.sub(r"‘", r"'", text, flags=re.MULTILINE)  # for standalone

    # WEIRD ENCODING: replace ’ with '
    text = re.sub(r"(\w)’(\w)", r"\1'\2", text, flags=re.MULTILINE)
    text = re.sub(r"’", r"'", text, flags=re.MULTILINE)  # for standalone
    
    # WEIRD ENCODING: replace weird ” and “ with " and accounts for letters before and after
    text = re.sub(r"(\w)”", r'\1"', text, flags=re.MULTILINE)
    text = re.sub(r"“(\w)", r'"\1', text, flags=re.MULTILINE)
    text = re.sub(r"”", r'"', text, flags=re.MULTILINE)  # for standalone
    text = re.sub(r"“", r'"', text, flags=re.MULTILINE)  # for standalone

    # SPACING: remove double spaces
    text = re.sub(r' {2,}', ' ', text, flags=re.MULTILINE)

    # SPACING: remove leading newlines
    text = re.sub(r'^\s*$', r'', text, flags = re.MULTILINE)

    # SPACING: remove leading whitespace (e.g., " Text" -> "Text")
    text = re.sub(r'^\s+', r'', text, flags=re.MULTILINE)

    #BASIC SPLITTING
    text = re.sub(r'^\d+ (\d+)', r'\1', text, flags = re.MULTILINE)
    text = re.sub(r'(\.\s)(\d+\s)', r'\1\n\2', text, flags=re.MULTILINE)
    text = re.sub(r'(\.\s)(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    
    text = re.sub(r'(\s)(\d+)([A-z])', r'\1\n\2\3', text, flags = re.MULTILINE)
    #for verses next to quotes
    text = re.sub(r'(\s)(\d+\")', r'\1\n\2', text, flags = re.MULTILINE)
    #next to single quotes to the right
    text = re.sub(r'\s(\d+\s\')', r'\n\1', text, flags = re.MULTILINE)
    #double quote before the new verse
    text = re.sub(r'(\")\s(\d+\s)', r'\1\n\2', text, flags = re.MULTILINE)
    
    #colon before new verse
    text = re.sub(r'(:)\s(\d+\s)', r'\1\n\2', text, flags = re.MULTILINE)
    #comma before new verse
    text = re.sub(r'(,)\s(\d+)', r'\1\n\2', text, flags = re.MULTILINE)
    # ; before new verse **********
    text = re.sub(r'(;)\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    # ) before new verse **********
    text = re.sub(r'(\))\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    # ] before new verse **********
    text = re.sub(r'(\])\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    #! before new verse
    text = re.sub(r'(!)\s(\d+)', r'\1\n\2', text, flags = re.MULTILINE)
    #? before new verse
    text = re.sub(r'(\?)\s(\d+)', r'\1\n\2', text, flags = re.MULTILINE)
    #— before new verse
    text = re.sub(r'(—)\s(\d+)', r'\1\n\2', text, flags = re.MULTILINE)

    # add space between number and capital letter **********
    text = re.sub(r'(\d+)([A-Z])', r'\1 \2', text, flags=re.MULTILINE)
    
    #adding tabs for splitting
    text = re.sub(r'(\d+-\d+)', r'\n\1', text, flags=re.MULTILINE)
    text = re.sub(r'(\d+[A-z]-\d+)', r'\1', text, flags = re.MULTILINE)
    
    # handling 12,13 in tuwali-ifugao
    text = re.sub(r'(\d+,\d+)', r'\1|', text, flags = re.MULTILINE)

    #split verse ranges from text first
    text = re.sub(r'^(\d+-\d+)', r'\1|', text, flags = re.MULTILINE)
    text = re.sub(r'^(\d+[A-z]-\d+)', r'\1|', text, flags = re.MULTILINE)

    # ( before new verse **********
    text = re.sub(r'(\d+)[\)\(]', r'\1|', text, flags=re.MULTILINE)
    # " before new verse **********
    text = re.sub(r'(\")\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    text = re.sub(r'(\")(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # ver w/o space
    text = re.sub(r'\s(\")(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # ver w/o space
    # [ before new verse **********
    text = re.sub(r'(\[)\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    text = re.sub(r'(\[)(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # ver w/o space
    
    #splitting the rest of it with tabs
    text = re.sub(r'^(\d+)([A-z][^-])', r'\1|\2', text, flags = re.MULTILINE)
    text = re.sub(r'^(\d+)\s', r'\1|', text, flags = re.MULTILINE)
    text = re.sub(r'^(\d+)\s(\")', r'\1|\2', text, flags = re.MULTILINE)
    text = re.sub(r'^(\d+)([\"\'])', r'\1|\2', text, flags = re.MULTILINE)
    # for Matthew|7|24 "Everyone... in english
    text = re.sub(r'^(\d+)\s(")', r'\1|\2', text, flags=re.MULTILINE) # ****** ADDED ^
    
    # remove empty lines
    text = re.sub(r'^\n', r'', text, flags=re.MULTILINE)
    # remove lines that only contain [ ********** TODO
    text = re.sub(r'(\d+\|)(\[)', r'\1', text, flags=re.MULTILINE)
    # remove lines that have nothing after |juan|8|8| ******
    text = re.sub(r'^(\w+\|\d+\|\w+\|\s*$)', r'', text, flags=re.MULTILINE)
    
    # ----- OTHER SPECIFIC QUIRKS BELOW -----
    # spanish
    text = re.sub(r'¶\s', r'|', text, flags = re.MULTILINE) # ****** removed ^
    text = re.sub(r'¶', r'|', text, flags = re.MULTILINE) # ****** ADDED no space ver^

    # handling upside down ? and !
    text = re.sub(r'^(\d+)(\()', r'\1|\2', text, flags=re.MULTILINE) # ****** ADDED ^
    text = re.sub(r'^(\d+)(\¿)', r'\1|\2', text, flags = re.MULTILINE) # ****** ADDED ^
    text = re.sub(r'^(\d+)(\¡)', r'\1|\2', text, flags = re.MULTILINE) # ****** ADDED ^
    text = re.sub(r'^(\d+)(\?)', r'\1|\2', text, flags = re.MULTILINE) # ****** ADDED ^

    # handling (39An in bikol and Matay|4|40) in yami 
    text = re.sub(r'(\()(\d+[A-z ])', r'\1\n\2', text, flags = re.MULTILINE) # text = re.sub(r'(\()(\d+[^\)])', r'\1\n \2', text, flags = re.MULTILINE) **********
    text = re.sub(r'(\[)(\d+[^\]])', r'\1\n\2', text, flags=re.MULTILINE) # and [9Pagkabuhay in bikol
    text = re.sub(r'(\' )(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # and 7'For in english and ' 24 "Everyone, in english **********removed space after \2
    text = re.sub(r'(\" )(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # for 29|(And all the people in english **********removed space after \2
    text = re.sub(r'^(\d+)(\')', r'\1|\2', text, flags=re.MULTILINE) # for 26'I in english and 6'Sinasabihan in bikol ********** # ****** ADDED ^
    # for John|8|11| ... again."]] 12 Again Jesus ... in english **********
    text = re.sub(r'(\]\]|\\"|\')\s+(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    # for Luke|2|22| ... Lord 23|(as it ... **********
    text = re.sub(r'(\d+)\|\(', r'\n\1 (', text, flags=re.MULTILINE)
    # for Luke|2|23 (as in english **********
    text = re.sub(r'^(\d+)\s(\()', r'\1|\2', text, flags=re.MULTILINE) # ****** ADDED ^

    # remove lines that have nothing after it or whitespace |juan|8|8| AGAIN ******
    text = re.sub(r'^(\w+\|\d+\|\w+\|\s*$)', r'', text, flags=re.MULTILINE)

    # remove 000| if not at the start of the line ********** 
    text = re.sub(r'000\|', r'000', text, flags=re.MULTILINE)

    # remove (number| ********** for yami
    text = re.sub(r'\((\d+)\|', r'\1', text, flags=re.MULTILINE) # text = re.sub(r'^(\w+\|\d+\|\d+\|.*)(\(\d+\|)', r'\1\2', text, flags=re.MULTILINE)

    # remove pipe in 3|B but after the first few pipes ********** (\w+\|\d+\|\d+\|.*)(\d+)\|([A-Z]) for yami **********
    text = re.sub(r'(\w+\|\d+\|\d+\|.*)(\d+)\|([A-Z].*?)', r'\1\2\3', text, flags=re.MULTILINE)
    text = re.sub(r'(\w+\|\d+\|\d+\|.*)(\d+)\|([A-Z].*?)', r'\1\2\3', text, flags=re.MULTILINE) # lowercase ver

    # add new line to make sure 44| and 55| are accounted for **********
    text = re.sub(r'\s(\d+\|)', r'\n\1', text)

    # for Luke|2|23 (as in english but for cebuano this time AGAIN **********
    text = re.sub(r'^(\d+)\s(\()', r'\1|\2', text, flags=re.MULTILINE) # ****** ADDED ^
    # for Matthew|7|24 " in english but for cebuano this time AGAIN **********
    text = re.sub(r'^(\d+)\s(\")', r'\1|\2', text, flags=re.MULTILINE) # ****** 

    # for 10,11|ot in tuwali_ifugao **********
    # reference text = re.sub(r'(\d+,\d+)', r'\1|', text, flags = re.MULTILINE)
    text = re.sub(r'(\d+,\d+\|)', r'\n\1', text, flags = re.MULTILINE)

    # " before new verse AGAIN **********
    text = re.sub(r'(\")\s(\d+)', r'\1\n\2', text, flags=re.MULTILINE)
    text = re.sub(r'(\")(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # ver w/o space
    text = re.sub(r'\s(\")(\d+)', r'\1\n\2', text, flags=re.MULTILINE) # ver w/o space

    # formatting for csv/xlsx, delimiter is | or pipe
    finalText = []
    bookCh = ""
    
    for line in text.splitlines():
        if line:
            if line[0] not in ["0","1","2","3","4","5","6","7","8","9"]:
                bookCh = line.strip()
            else:
                line = bookCh + "|" +line
        finalText.append(line)


    text= '\n'.join(finalText)

    text = re.sub(r'^([SMJLY].*)\d+(\n)', r'', text, flags = re.MULTILINE)
    #text = re.sub(r'^((\w+) (\d+))', r"\2|\3", text, flags=re.MULTILINE)
    text = re.sub(r'^([SMJLY][^|]*?)\s+(\d+)', r'\1|\2', text, flags=re.MULTILINE)
    
    text = "Book|Chapter #|Verse #|Verse\n" + text

    # remove lines that have nothing after it or whitespace |juan|8|8| AGAIN ******
    text = re.sub(r'^(\w+\|\d+\|\w+\|\s*$)', r'', text, flags=re.MULTILINE)

    # remove empty lines AGAIN **********
    text = re.sub(r'^\n', r'', text, flags=re.MULTILINE)

    return text

The following function, `divide_into_sentences` takes the raw lang.txt files and divides them into verses. The following steps are as follows:
<span style="background-color:red;color:white;">STEPS HERE</span>

In [5]:
def divide_into_sentences(text):
    # remove any numbers with spaces at the start of lines
    # example: "1 This is a verse -> "This is a verse"
    text = re.sub(r'^\d+\s*', r'', text, flags=re.MULTILINE)
    
    # removes book and chapter headings (e.g., "Matthew 1")
    # ^[A-Za-zÀ-ÖØ-öø-ÿ\s]+ matches the book name (also covers accented letters for other languaes that do have them)
    # \s+ matches the space between the book name and chapter number
    # \d+ matches the chapter number
    # \s*\n? matches any trailing spaces and newline
    text = re.sub(r'^[0-9A-Za-zÀ-ÖØ-öø-ÿ\s]+\d+\s*\n?', '', text, flags=re.MULTILINE)

    # removes verse numbers
    # (?![\d,\.]) makes sure we don't remove numbers that are part of decimals or commas
    text = re.sub(r'\b\d+(?![\d,\.])', '', text)

    # replaces multiple newlines with a single newline
    text = re.sub(r'\n+', r'\n', text)

    # replaces multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)

    # ensures there is a space after sentence-ending punctuation if followed by a non-space character
    # [.!?] matches sentence ending punctuation
    # ["”’\'\)\]\}] matches any closing quotes, parentheses, or brackets that may follow the punctuation
    # (?=\S) ensures the punctuation is followed by a non-space character
    text = re.sub(r'([.!?]["”’\'\)\]\}]?)(?=\S)', r'\1 ', text)

    # remove spaces after opening punctuation
    text = re.sub(r'([“‘\(\[])\s+', r'\1', text)

    # remove spaces before closing punctuation
    text = re.sub(r'\s+([”’\)\]])', r'\1', text)

    # ensures each sentence starts on a new line
    # [.!?] matches sentence ending punctuation
    # ["\'”’\)\]\}] matches any closing quotes, parentheses, or brackets that may follow the punctuation
    # \s+ matches the whitespace following the punctuation
    text = re.sub(r'([.!?](?:["\'”’\)\]\}]+)?)\s+', r'\1\n', text)

    # This regex keeps letters (including accented), numbers, and common punctuation marks 
    # while removing inconsistent special characters.
    # Basically, if its not in the list of characters, it gets omitted.
    # I had to consult AI with this one in giving me the characters to be included.
    text = re.sub(r'[^A-Za-zÀ-ÖØ-öø-ÿ\u00C0-\u024F\u1E00-\u1EFF0-9\s\.\,\!\?\:\;\'"“”‘’\-\(\)]', "", text)
    
    # removes leading whitespace (e.g., " Text" -> "Text")
    text = re.sub(r'^\s+', r'', text, flags=re.MULTILINE)

    # brings closing parentheses to the previous line if they are on a new line
    text = re.sub(r'\n\)', r')\n', text)

    # removes white space at the start of each line after ", ', or )"
    text = re.sub(r'^([\"\'\)])\s+', r'\1', text, flags=re.MULTILINE)

    # removes leading whitespace (e.g., " Text" -> "Text")
    text = re.sub(r'^\s+', r'', text, flags=re.MULTILINE)

    # standardizes quotes to " and ' at the end
    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r'[‘’]', "'", text)

    return text

NOTE: Please delete the cleaned folder before clicking `Run All`. 

In [6]:
# data_folder = Path("data") already exists above
cleaned_folder = Path("data/cleaned")
cleaned_folder.mkdir(parents=True, exist_ok=True) 

We will run `divide_into_verses` on the raw webscraping .txts in `/data`. This will be used for the Excel sheets later in the project.

In [7]:
for lang in languages:
    data_folder = Path("data")
    file_path = data_folder / f"{lang}.txt"
    output_path = cleaned_folder / f"{lang}-cleaned.txt"

    # SKIP if the input file does not exist or is empty
    if not file_path.exists() or file_path.stat().st_size == 0:
        print(f"Skipped: {file_path} (file does not exist or is empty)")
        continue

    # DELETE IF EXISTS
    if output_path.exists():
        print(f"unlinking/deleting old version of {lang}-cleaned.txt")
        output_path.unlink()

    # read input
    with file_path.open("r", errors="ignore", encoding="utf-8") as f:
        text = f.read()

    # clean the text
    cleaned_text = divide_into_verses(text)

    # save
    with output_path.open("w", encoding="utf-8") as f:
        f.write(cleaned_text)

    print(f"Cleaned and saved: {output_path}")

unlinking/deleting old version of spanish-cleaned.txt
Cleaned and saved: data\cleaned\spanish-cleaned.txt
unlinking/deleting old version of tagalog-cleaned.txt
Cleaned and saved: data\cleaned\tagalog-cleaned.txt
unlinking/deleting old version of english-cleaned.txt
Cleaned and saved: data\cleaned\english-cleaned.txt
unlinking/deleting old version of hiligaynon-cleaned.txt
Cleaned and saved: data\cleaned\hiligaynon-cleaned.txt
unlinking/deleting old version of bikol-cleaned.txt
Cleaned and saved: data\cleaned\bikol-cleaned.txt
unlinking/deleting old version of waray-cleaned.txt
Cleaned and saved: data\cleaned\waray-cleaned.txt
unlinking/deleting old version of ilocano-cleaned.txt
Cleaned and saved: data\cleaned\ilocano-cleaned.txt
unlinking/deleting old version of cebuano-cleaned.txt
Cleaned and saved: data\cleaned\cebuano-cleaned.txt
unlinking/deleting old version of kapampangan-cleaned.txt
Cleaned and saved: data\cleaned\kapampangan-cleaned.txt
unlinking/deleting old version of pangas

Next, we will run `divide_into_sentences` on the raw data.

In [8]:
for lang in languages:
    data_folder = Path("data")
    file_path = data_folder / f"{lang}.txt"
    output_path = cleaned_folder / f"(sentences)-{lang}-cleaned.txt"

    # SKIP if the input file does not exist or is empty
    if not file_path.exists() or file_path.stat().st_size == 0:
        print(f"Skipped: {file_path} (file does not exist or is empty)")
        continue

    # DELETE IF EXISTS
    if output_path.exists():
        print(f"unlinking/deleting old version of {lang}-cleaned.txt")
        output_path.unlink()

    # read input
    with file_path.open("r", errors="ignore", encoding="utf-8") as f:
        text = f.read()

    # clean the text
    cleaned_text = divide_into_sentences(text)

    # save
    with output_path.open("w", encoding="utf-8") as f:
        f.write(cleaned_text)

    print(f"Cleaned and saved: {output_path}")

unlinking/deleting old version of spanish-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-spanish-cleaned.txt
unlinking/deleting old version of tagalog-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-tagalog-cleaned.txt
unlinking/deleting old version of english-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-english-cleaned.txt
unlinking/deleting old version of hiligaynon-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-hiligaynon-cleaned.txt
unlinking/deleting old version of bikol-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-bikol-cleaned.txt
unlinking/deleting old version of waray-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-waray-cleaned.txt
unlinking/deleting old version of ilocano-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-ilocano-cleaned.txt
unlinking/deleting old version of cebuano-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-cebuano-cleaned.txt
unlinking/deleting old version of kapampangan-cleaned.txt


Here, we will be getting the word count of the raw `lang.txt` files, freshly scraped from Bible.com, and then a word count of the `(sentences)-lang.txt` files.

In [9]:
data_folder = Path("data")
total_word_count = 0

print("PRE-CLEAN WORD COUNT")

for lang in languages:
    cleaned_file_path = data_folder / f"{lang}.txt"  

    if cleaned_file_path.exists() and cleaned_file_path.stat().st_size > 0:
        with cleaned_file_path.open("r", encoding="utf-8") as f:
            text = f.read()
            word_count = len(text.split())
            total_word_count += word_count
            print(f"Word count for {lang}.txt: {word_count}")
    else:
        print(f"Skipped: {cleaned_file_path} (file does not exist or is empty)")

print(f"Total word count across all -cleaned files: {total_word_count}")

PRE-CLEAN WORD COUNT
Word count for spanish.txt: 76085
Word count for tagalog.txt: 86888
Word count for english.txt: 81210
Word count for hiligaynon.txt: 92970
Word count for bikol.txt: 79624
Word count for waray.txt: 89081
Word count for ilocano.txt: 73149
Word count for cebuano.txt: 84973
Word count for kapampangan.txt: 84635
Word count for pangasinense.txt: 77748
Word count for yakan.txt: 84123
Word count for ivatan.txt: 97534
Word count for tausug.txt: 113208
Word count for yami.txt: 107967
Word count for tuwali_ifugao.txt: 82580
Word count for masbateno.txt: 88302
Total word count across all -cleaned files: 1400077


In [10]:
data_folder = Path("data/cleaned")
total_word_count = 0

print("SENTENCE WORD COUNT")

for lang in languages:
    sentence_file_path = data_folder / f"(sentences)-{lang}-cleaned.txt"  

    if sentence_file_path.exists() and sentence_file_path.stat().st_size > 0:
        with sentence_file_path.open("r", encoding="utf-8") as f:
            text = f.read()
            word_count = len(text.split())
            total_word_count += word_count
            print(f"Word count for {lang}-cleaned.txt: {word_count}")
    else:
        print(f"Skipped: {sentence_file_path} (file does not exist or is empty)")

print(f"Total word count across all (sentences)-cleaned files: {total_word_count}")

SENTENCE WORD COUNT
Word count for spanish-cleaned.txt: 75318
Word count for tagalog-cleaned.txt: 86596
Word count for english-cleaned.txt: 80426
Word count for hiligaynon-cleaned.txt: 92876
Word count for bikol-cleaned.txt: 79543
Word count for waray-cleaned.txt: 88941
Word count for ilocano-cleaned.txt: 73030
Word count for cebuano-cleaned.txt: 84773
Word count for kapampangan-cleaned.txt: 84254
Word count for pangasinense-cleaned.txt: 77670
Word count for yakan-cleaned.txt: 83832
Word count for ivatan-cleaned.txt: 97250
Word count for tausug-cleaned.txt: 113228
Word count for yami-cleaned.txt: 107672
Word count for tuwali_ifugao-cleaned.txt: 82321
Word count for masbateno-cleaned.txt: 88228
Total word count across all (sentences)-cleaned files: 1395958


We can get the word count for the sentences here.

SUMMARY:
| Target Language      | Link to Source      | # Words Before Cleaning | # Words After Cleaning |
| ------------- | ------------- | ------------- |  ------------- |
| Spanish  | https://www.bible.com/versions/1076-jbs-biblia-del-jubileo | 76085 | 75318 |
| Tagalog | https://www.bible.com/versions/177-tlab-ang-biblia | 86888 |86596  |
| English | https://www.bible.com/versions/3523-nrsvue-new-revised-standard-version-updated-edition-2021 |  81210 | 80426 |
| Hiligaynon/Ilonggo | https://www.bible.com/versions/2190-mbbhil12-maayong-balita-nga-biblia-2012 | 92970 | 92876 |
| Bikol/Bikolano | https://www.bible.com/versions/890-mbbbik92-marahay-na-bareta-biblia | 79624 | 79543 |
| Waray | https://www.bible.com/versions/2198-mbbsam-samarenyo-meaning-based-bible-1984 | 89081 | 88941 |
| Ilocano | https://www.bible.com/versions/782-ripv-ti-baro-a-naimbag-a-damag-biblia | 73149 | 73030 |
| Cebuano | https://www.bible.com/versions/562-rcpv-ang-bag-ong-maayong-balita-biblia | 84973 | 84773 |
| Kapampangan | https://www.bible.com/versions/1141-pmpv-ing-mayap-a-balita-biblia | 84635 | 84254 |
| Pangasinense | https://www.bible.com/versions/2194-mbbpan83-maung-a-balita-biblia | 77748 | 77670 |
| Yakan | https://www.bible.com/versions/1388-yakv-yakan | 84123 | 83832 |
| Ivatan | https://www.bible.com/versions/1315-vtsp-ivatan | 97534 | 97250 |
| Tausug  | https://www.bible.com/versions/1319-tsg-kitab-injil | 113208 | 113228 |
| Yami  | https://www.bible.com/versions/2364-snt-seysyo-no-tao | 107967 | 107672 |
| Tuwali Ifugao | https://www.bible.com/versions/2123-ifkwb-nan-kalin-apu-dios | 82580 | 82321 |
| Masbateño/Masbatenyo | https://www.bible.com/versions/1222-msb-masbatenyo | 88302 | 88228 |
| TOTAL | 16 | 1400077 | 1395958 |

## 3. Parallel Corpus
Instructions: *Create a parallel corpus organized by verse. The number of language pairs should be equal to 3 + n, where n is the number of group member. Example: TGL-CEB, CEB-ILO, ILO-WAR, TGL-WAR.*

Using the verse collection called `all-languages-cleaned-verses.xlsx`, which manually imported the (lang)-cleaned.txt files with the delimiter set to pipe (`|`), we will now be creating the parallel corpora.

For our implementation, we have selected the following languages: english tagalog yami kapampangan pangasinense
- english-tagalog (ENG-TGL)
- english-yami (ENG-TAO)
- english-kapampangan (ENG-PAM)
- tagalog-kapampangan (TGL-PAM)
- tagalog-yami (TGL-TAO)
- tagalog-pangasinense (TGL-PAG)
- kapampangan-yami (PAM-TAO)

In [11]:
# CREATE CLEANED VERSES
# data_folder = Path("data/cleaned")
output_excel_path = Path("cleaned_verses.xlsx")
csv_folder = Path("data/csv")

# if cleaned_verses.xlsx exists just delete it
if output_excel_path.exists():
    print("unlinking/deleting old version of cleaned_verses.xlsx")
    output_excel_path.unlink()

languages = ["spanish", "tagalog", "english", "hiligaynon", "bikol", "waray", "ilocano", "cebuano", "kapampangan", "pangasinense", "yakan", "ivatan", "tausug", "yami", "tuwali_ifugao", "masbateno"]

# cleaned_verses
with pd.ExcelWriter(output_excel_path) as pd_writer:
    for lang in languages:
        print(f"attempting {lang}")
        cleaned_file_path = data_folder / f"{lang}-cleaned.txt"

        if cleaned_file_path.exists() and cleaned_file_path.stat().st_size > 0:
            try: 
                with cleaned_file_path.open("r", encoding="utf-8") as f:
                    df = pd.read_csv(cleaned_file_path, sep="\\|", on_bad_lines="warn", engine="python", quoting=3)
                    excel = df.to_excel(pd_writer, sheet_name=f'{lang}-verses', index=False, engine="python") # false indexing bc we already have it
                    print(f"Added {lang} to cleaned_verses.xlsx")
            except Exception as e:
                print(f"Error processing {lang}: {e}")

unlinking/deleting old version of cleaned_verses.xlsx
attempting spanish
Added spanish to cleaned_verses.xlsx
attempting tagalog
Added tagalog to cleaned_verses.xlsx
attempting english
Added english to cleaned_verses.xlsx
attempting hiligaynon
Added hiligaynon to cleaned_verses.xlsx
attempting bikol
Added bikol to cleaned_verses.xlsx
attempting waray
Added waray to cleaned_verses.xlsx
attempting ilocano
Added ilocano to cleaned_verses.xlsx
attempting cebuano
Added cebuano to cleaned_verses.xlsx
attempting kapampangan
Added kapampangan to cleaned_verses.xlsx
attempting pangasinense
Added pangasinense to cleaned_verses.xlsx
attempting yakan
Added yakan to cleaned_verses.xlsx
attempting ivatan
Added ivatan to cleaned_verses.xlsx
attempting tausug
Added tausug to cleaned_verses.xlsx
attempting yami
Added yami to cleaned_verses.xlsx
attempting tuwali_ifugao
Added tuwali_ifugao to cleaned_verses.xlsx
attempting masbateno
Added masbateno to cleaned_verses.xlsx


In [12]:
# CREATE PARALLEL CORPORA
# data_folder = Path("data/cleaned")
output_excel_path = Path("parallel_corpora.xlsx")
csv_folder = Path("data/csv")

# if parallel_corpora.xlsx exists just delete it
if output_excel_path.exists():
    print("unlinking/deleting old version of parallel_corpora.xlsx")
    output_excel_path.unlink()

corpora_lang = ["english", "tagalog", "yami", "kapampangan", "pangasinense"]

# parallel corpora
with pd.ExcelWriter(output_excel_path) as pd_writer:
    for lang in corpora_lang:
        print(f"attempting {lang}")
        cleaned_file_path = data_folder / f"{lang}-cleaned.txt"

        if cleaned_file_path.exists() and cleaned_file_path.stat().st_size > 0:
            try: 
                with cleaned_file_path.open("r", encoding="utf-8") as f:
                    df = pd.read_csv(cleaned_file_path, sep="\\|", on_bad_lines="warn", engine="python", quoting=3)
                    excel = df.to_excel(pd_writer, sheet_name=f'{lang}-verses', index=False, engine="python") # false indexing bc we already have it
                    print(f"Added {lang} to parallel_corpora.xlsx")
            except Exception as e:
                print(f"Error processing {lang}: {e}")

print("Finished")

unlinking/deleting old version of parallel_corpora.xlsx
attempting english
Added english to parallel_corpora.xlsx
attempting tagalog
Added tagalog to parallel_corpora.xlsx
attempting yami
Added yami to parallel_corpora.xlsx
attempting kapampangan
Added kapampangan to parallel_corpora.xlsx
attempting pangasinense
Added pangasinense to parallel_corpora.xlsx
Finished


From the excel sheets, we'll import them into dataframes for each of the languages.

In [51]:
corpora_lang = ["english", "tagalog", "yami", "kapampangan", "pangasinense"]

eng_df = pd.read_excel("parallel_corpora.xlsx", sheet_name="english-verses")
tgl_df = pd.read_excel("parallel_corpora.xlsx", sheet_name="tagalog-verses")
tao_df = pd.read_excel("parallel_corpora.xlsx", sheet_name="yami-verses")
pam_df = pd.read_excel("parallel_corpora.xlsx", sheet_name="kapampangan-verses")
pag_df = pd.read_excel("parallel_corpora.xlsx", sheet_name="pangasinense-verses")

dataframes = [ eng_df, tgl_df, tao_df, pam_df, pag_df ]

# rename verse cols https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html
eng_df = eng_df.rename(columns={'Verse': 'English'})
tgl_df = tgl_df.rename(columns={'Verse': 'Tagalog'})
tao_df = tao_df.rename(columns={'Verse': 'Yami'})
pam_df = pam_df.rename(columns={'Verse': 'Kapampangan'})
pag_df = pag_df.rename(columns={'Verse': 'Pangasinense'})

# handle dtypes
for i, df in enumerate(dataframes):
    for col in ['Book', 'Chapter #', 'Verse #', 'Verse']:
        if col in df.columns:
            dataframes[i][col] = df[col].astype(object)

for df in dataframes:
    print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3768 entries, 0 to 3767
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Book       3768 non-null   object
 1   Chapter #  3768 non-null   object
 2   Verse #    3768 non-null   object
 3   Verse      3768 non-null   object
dtypes: object(4)
memory usage: 117.9+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3779 entries, 0 to 3778
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Book       3779 non-null   object
 1   Chapter #  3779 non-null   object
 2   Verse #    3779 non-null   object
 3   Verse      3779 non-null   object
dtypes: object(4)
memory usage: 118.2+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3770 entries, 0 to 3769
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Book       3770 non-null   object
 1 

In [52]:
# line counter
print(len(eng_df))
print(len(tgl_df))
print(len(tao_df))
print(len(pam_df))
print(len(pag_df))

3768
3779
3770
3776
3763


After some digging, we found out that there are some lines that are really missing on the website. (Ex. English literally has no Matthew 17:21) Using this [source](https://www.biblememorygoal.com/how-many-chapters-verses-in-the-bible/), we can find how many verses are actually expected. We can edit the dataframes and excel sheets as shown:

In [ ]:
# TODO: as defined earlier
chapterRanges = {
    "mat": [1, 28],
    "mrk": [1, 16],
    "luk": [1, 24],
    "jhn": [1, 21]
}
verseRanges = {
    "mat": 1071,
    "mrk": 678,
    "luk": 1151,
    "jhn": 879
}

# TODO: check if verseRange per chapter is the same


Checked 28 chapters — total missing verses: 1
Checked 28 chapters — total missing verses: 0
Checked 28 chapters — total missing verses: 0
Checked 28 chapters — total missing verses: 0
Checked 28 chapters — total missing verses: 2


In [ ]:
# english-tagalog (ENG-TGL)
eng_tgl = pd.merge(
    eng_df[['Chapter #', 'Verse #', 'English']], 
    tgl_df[['Chapter #', 'Verse #', 'Tagalog']], 
    on=['Chapter #', 'Verse #'], 
    how='outer'
)
eng_tgl

,Chapter #,Verse #,English,Tagalog
0,1,1,An account of the genealogy of Jesus the Messi...,"Ang aklat ng lahi ni Jesucristo, na anak ni Da..."
1,1,1,An account of the genealogy of Jesus the Messi...,"Ang pasimula ng evangelio ni Jesucristo, ang A..."
2,1,1,An account of the genealogy of Jesus the Messi...,Yamang marami ang nagpilit mag-ayos ng isang k...
3,1,1,An account of the genealogy of Jesus the Messi...,"Nang pasimula siya ang Verbo, at ang Verbo ay ..."
4,1,1,The beginning of the good news of Jesus Christ.,"Ang aklat ng lahi ni Jesucristo, na anak ni Da..."
...,...,...,...,...
11785,28,16,"Now the eleven disciples went to Galilee, to t...",Datapuwa't nagsiparoon ang labingisang alagad ...
11786,28,17,"When they saw him, they worshiped him, but the...","At nang siya'y kanilang makita, ay kanilang si..."
11787,28,18,"And Jesus came and said to them, ""All authorit...",At lumapit si Jesus sa kanila at sila'y kaniya...
11788,28,19,Go therefore and make disciples of all nations...,"Dahil dito magsiyaon nga kayo, at gawin ninyon..."


In [ ]:
# english-yami (ENG-TAO)
eng_tao = pd.merge(
    eng_df[['Chapter #', 'Verse #', 'English']], 
    tao_df[['Chapter #', 'Verse #', 'Yami']], 
    on=['Chapter #', 'Verse #'], 
    how='inner'
)
eng_tao

ValueError: You are trying to merge on int64 and object columns for key 'Verse #'. If you wish to proceed you should use pd.concat

In [ ]:
# english-kapampangan (ENG-PAM)
eng_pam = pd.merge(
    eng_df[['Chapter #', 'Verse #', 'English']], 
    pam_df[['Chapter #', 'Verse #', 'Kapampangan']], 
    on=['Chapter #', 'Verse #'], 
    how='inner'
)
eng_pam

ValueError: You are trying to merge on int64 and object columns for key 'Verse #'. If you wish to proceed you should use pd.concat

In [ ]:
# tagalog-kapampangan (TGL-PAM)
tgl_pam = pd.merge(
    tgl_df[['Chapter #', 'Verse #', 'Tagalog']], 
    pam_df[['Chapter #', 'Verse #', 'Kapampangan']], 
    on=['Chapter #', 'Verse #'], 
    how='inner'
)
tgl_pam

,Chapter #,Verse #,Tagalog,Kapampangan
0,1,1,1,1
1,1,1,1,1
2,1,1,1,1
3,1,1,1,1
4,1,2,2,2
...,...,...,...,...
11797,21,24,24,24
11798,21,24,24,24
11799,21,25,25,25
11800,21,25,25,25


In [ ]:
# tagalog-yami (TGL-TAO)
tgl_tao = pd.merge(
    tgl_df[['Chapter #', 'Verse #', 'Tagalog']], 
    tao_df[['Chapter #', 'Verse #', 'Yami']], 
    on=['Chapter #', 'Verse #'], 
    how='inner'
)
tgl_tao

,Chapter #,Verse #,Tagalog,Yami
0,1,1,1,1
1,1,1,1,1
2,1,1,1,1
3,1,1,1,1
4,1,2,2,2
...,...,...,...,...
11766,21,24,24,24
11767,21,24,24,24
11768,21,25,25,25
11769,21,25,25,25


In [ ]:
# tagalog-pangasinense (TGL-PAG)
tgl_pag = pd.merge(
    tgl_df[['Chapter #', 'Verse #', 'Tagalog']], 
    pag_df[['Chapter #', 'Verse #', 'Pangasinense']], 
    on=['Chapter #', 'Verse #'], 
    how='inner'
)
tgl_pag

,Chapter #,Verse #,Tagalog,Pangasinense
0,1,1,1,1
1,1,1,1,1
2,1,1,1,1
3,1,1,1,1
4,1,2,2,2
...,...,...,...,...
11771,21,24,24,24
11772,21,24,24,24
11773,21,25,25,25
11774,21,25,25,25


In [ ]:
# kapampangan-yami (PAM-TAO)
pam_tao = pd.merge(
    pam_df[['Chapter #', 'Verse #', 'Kapampangan']], 
    tao_df[['Chapter #', 'Verse #', 'Yami']], 
    on=['Chapter #', 'Verse #'], 
    how='inner'
)
pam_tao

,Chapter #,Verse #,Kapampangan,Yami
0,1,1,1,1
1,1,1,1,1
2,1,1,1,1
3,1,1,1,1
4,1,2,2,2
...,...,...,...,...
11755,21,24,24,24
11756,21,24,24,24
11757,21,25,25,25
11758,21,25,25,25


Finally, we can turn them back into excel files or add them back to parallel_corpora

## X. AI Declaration


## X. Resources